# Math Escalation — Kaggle LLM Training Demo

This notebook trains a real LLM using **GRPO + Unsloth** on adaptive math problems.
It is **fully self-contained** — it generates problems and verifies rewards internally,
so you don't need a running server.

This Kaggle version is tuned for an accuracy-improving GPU demo:
- **Qwen2.5-0.5B-Instruct** with 4-bit quantization
- **LoRA r=32 / α=64** — bigger adapter capacity to actually shift model behavior
- **150 GRPO steps × 6 generations** — denser advantage signal per prompt
- **20 deterministic eval cases** (fixed seed) for stable before/after scoring
- **Robust answer extraction** — number after `</thought>`, with `\boxed{}` and last-number fallbacks
- **Correctness-weighted reward** — +2.0 right / −1.0 wrong dominates a tiny format bonus
- **No repetition_penalty at eval** — math digits naturally repeat, the penalty was hurting accuracy

In [ ]:
# Cell 1 — Install Dependencies
# Internet must be enabled in Kaggle notebook settings.
!pip install -q -U "unsloth[colab-new]" "trl>=0.12.0" datasets
print('Dependencies installed!')

In [ ]:
# Cell 2 — Embedded Math Environment (No Server Needed)
import random
import re
import math

def generate_problem(difficulty: int):
    """Generate a math problem for the given tier (1-10) and return (problem_str, answer)."""
    d = difficulty
    if d == 1:
        a, b = random.randint(1, 10), random.randint(1, 10)
        return f"{a} + {b}", float(a + b)
    elif d == 2:
        a, b = random.randint(10, 99), random.randint(10, 99)
        return f"{a} + {b}", float(a + b)
    elif d == 3:
        a, b = sorted([random.randint(10, 99), random.randint(10, 99)], reverse=True)
        return f"{a} - {b}", float(a - b)
    elif d == 4:
        a, b = random.randint(2, 12), random.randint(2, 12)
        return f"{a} * {b}", float(a * b)
    elif d == 5:
        a, b, c = random.randint(2, 10), random.randint(2, 10), random.randint(1, 20)
        return f"{a} * {b} + {c}", float(a * b + c)
    elif d == 6:
        a, b, c = random.randint(2, 10), random.randint(2, 10), random.randint(1, 20)
        return f"{a} * ({b} + {c})", float(a * (b + c))
    elif d == 7:
        a, b = random.randint(5, 20), random.randint(2, 9)
        return f"{a * b} / {b}", float(a)
    elif d == 8:
        a, x = random.randint(2, 9), random.randint(1, 15)
        b = random.randint(1, 30)
        c = a * x + b
        return f"Solve for x: {a}x + {b} = {c}", float(x)
    elif d == 9:
        x = random.randint(2, 20)
        return f"sqrt({x * x})", float(x)
    else:
        a, x_val, div = random.randint(2, 5), random.randint(1, 10), random.randint(2, 4)
        b = div * random.randint(2, 8) - a * x_val
        e = (a * x_val + b) // div
        return f"Solve for x: ({a}x + {b}) / {div} = {e}", float(x_val)

def check_answer(problem_str: str, expected: float, model_answer: float) -> float:
    """Returns reward: +1.0 correct, -0.5 wrong."""
    return 1.0 if abs(model_answer - expected) < 1e-4 else -0.5

print('✅ Math environment ready! Tiers 1-10 embedded.')

In [ ]:
# Cell 3 — Load Model with Unsloth (4-bit quantized for Kaggle GPU)
from unsloth import FastLanguageModel
import torch

# 768 leaves room for a longer prompt + 256-token completion during GRPO rollouts.
max_seq_length = 768
dtype = None
load_in_4bit = True
MODEL_NAME = 'unsloth/Qwen2.5-0.5B-Instruct'

print(f'Loading {MODEL_NAME} with Unsloth 4-bit...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    fast_inference = False,
)

# Bigger LoRA (r=32, alpha=64) gives the policy enough capacity to actually
# learn the math-format -> right-answer mapping in 150 GRPO steps.
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

print(f'Model loaded! Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 4 — Correctness-Weighted Reward Function (with robust extractor)
import re

_NUM_RE = re.compile(r'-?\d+\.?\d*')

def extract_predicted_number(text: str):
    """Robust answer extraction.

    Strategy (in order):
      1. If the model wrote `</thought>`, take the FIRST number after it
         (this is the model's committed final answer, by prompt design).
      2. Otherwise, strip any `<thought>...</thought>` blocks and take the
         LAST number — most CoT outputs end with the final answer.
      3. Look for `\\boxed{...}` if the model used LaTeX style.
    """
    boxed = re.search(r'\\boxed\{\s*(-?\d+\.?\d*)\s*\}', text)
    if boxed:
        try:
            return float(boxed.group(1))
        except ValueError:
            pass

    if '</thought>' in text:
        after = text.split('</thought>', 1)[1]
        nums = _NUM_RE.findall(after)
        if nums:
            try:
                return float(nums[0])
            except ValueError:
                pass

    clean = re.sub(r'<thought>.*?</thought>', '', text, flags=re.DOTALL)
    nums = _NUM_RE.findall(clean)
    if nums:
        try:
            return float(nums[-1])
        except ValueError:
            return None
    return None

def compute_reward(*args, **kwargs):
    """TRL GRPO reward function.
    Dataset answers arrive as kwargs['answer']; signature is compatible with newer TRL.

    Reward composition (per sample):
      env_reward     : +2.0 correct / -1.0 wrong  (dominant signal)
      numeric_reward : +0.1 if any number produced / -0.2 if none
      format_reward  : up to +0.15 for using <thought>...</thought>
                       (only counts if a numeric answer was also produced,
                        so the model can't game format alone)
    """
    prompts = kwargs.get('prompts')
    completions = kwargs.get('completions')
    answers = kwargs.get('answer') or kwargs.get('answers')
    if len(args) >= 3 and answers is None:
        answers = args[2]
    if len(args) >= 2 and (prompts is None or completions is None):
        completions, prompts = args[0], args[1]
    if answers is None:
        raise ValueError(f"Missing answer column. kwargs={sorted(kwargs.keys())}")

    rewards = []
    for completion, _prompt, expected in zip(completions, prompts, answers):
        text = completion[0]['content'] if isinstance(completion, list) else str(completion)

        predicted = extract_predicted_number(text)
        numeric_reward = 0.1 if predicted is not None else -0.2

        env_reward = -1.0
        if predicted is not None:
            try:
                expected_val = float(expected)
                env_reward = 2.0 if abs(predicted - expected_val) < 1e-4 else -1.0
            except Exception:
                env_reward = -1.0

        format_reward = 0.0
        if predicted is not None and '<thought>' in text and '</thought>' in text:
            format_reward = 0.15

        rewards.append(env_reward + numeric_reward + format_reward)

    return rewards

print('Reward function ready: +2.0 correct / -1.0 wrong, robust last-number extractor with boxed{} support.')

In [ ]:
# Cell 5 — Build Curriculum Dataset (Embedded, No Server)
import datasets

def build_curriculum_dataset(n_problems=200):
    """Build a compact curriculum dataset for a fast Kaggle demo run."""
    print(f'Building curriculum dataset with {n_problems} problems...')
    data = []
    tier_counts = {
        1: 30, 2: 30, 3: 25, 4: 25, 5: 20,
        6: 20, 7: 15, 8: 15, 9: 10, 10: 10
    }

    for tier, count in tier_counts.items():
        for _ in range(count):
            problem_str, answer = generate_problem(tier)
            prompt = (
                f'Solve the following math problem. '
                f'Think step-by-step inside <thought> tags, '
                f'then give ONLY the final numeric answer.\n\n'
                f'[Tier {tier}/10] Problem: {problem_str}\n'
                f'Answer:'
            )
            data.append({'prompt': prompt, 'answer': str(answer)})

    random.shuffle(data)
    ds = datasets.Dataset.from_list(data)
    print(f'Dataset ready: {len(ds)} problems across Tiers 1-10')
    sample = ds[0]
    print(f"Sample prompt: {sample['prompt'][:160]}...")
    print(f"Expected answer: {sample['answer']}")
    return ds

train_dataset = build_curriculum_dataset(200)
print(f'Dataset size: {len(train_dataset)}')

In [ ]:
# Cell 6 — Baseline Evaluation Before Training (20 deterministic cases)
FastLanguageModel.for_inference(model)

# Deterministic eval set so baseline and post-training are scored on
# the EXACT same problems — improvement (or regression) is meaningful.
EVAL_TIER_DISTRIBUTION = (
    [1] * 4 + [2] * 4 + [3] * 3 + [4] * 3 + [5] * 2 + [6] * 2 + [7] * 1 + [9] * 1
)
assert len(EVAL_TIER_DISTRIBUTION) == 20

_eval_rng_state = random.getstate()
random.seed(42)
EVAL_CASES = []
for _tier in EVAL_TIER_DISTRIBUTION:
    _p, _ans = generate_problem(_tier)
    EVAL_CASES.append((_p, _tier, _ans))
random.setstate(_eval_rng_state)

print(f'Eval set: {len(EVAL_CASES)} deterministic problems (seed=42)')
for _p, _t, _a in EVAL_CASES[:4]:
    print(f'  [Tier {_t}] {_p} = {_a}')
print('  ...')

def test_model(problem_str, tier=1):
    """Greedy decode without repetition_penalty (math digits naturally repeat)."""
    prompt = (
        f'Solve the following math problem. '
        f'Think step-by-step inside <thought> tags, '
        f'then give ONLY the final numeric answer.\n\n'
        f'[Tier {tier}/10] Problem: {problem_str}\n'
        f'Answer:'
    )
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    inputs = tokenizer([prompt], return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def evaluate_model(label):
    """Score the model on EVAL_CASES using the robust extract_predicted_number."""
    correct = 0
    print(f'\n{label}')
    print('=' * 60)
    for problem, tier, expected in EVAL_CASES:
        response = test_model(problem, tier)
        predicted = extract_predicted_number(response)
        ok = predicted is not None and abs(predicted - expected) < 1e-4
        correct += int(ok)
        status = 'PASS' if ok else 'FAIL'
        print(f'{status} [{tier}] {problem} = {expected} | Model: {predicted}')
    score = 100 * correct / len(EVAL_CASES)
    print('-' * 60)
    print(f'{label} Score: {correct}/{len(EVAL_CASES)} ({score:.1f}%)')
    return correct, score

baseline_correct, baseline_score = evaluate_model('Baseline Evaluation Before GRPO')

In [ ]:
# Cell 7 — GRPO Training Configuration & Launch
from trl import GRPOConfig, GRPOTrainer

FastLanguageModel.for_training(model)

# 150 steps + 6 generations per prompt gives GRPO denser advantage signal
# than 120 steps x 4 gens. Longer completions (256) prevent CoT truncation
# on tier 8-10 problems, which used to silently hurt correctness reward.
training_args = GRPOConfig(
    output_dir = './math_escalation_grpo',
    num_train_epochs = 1,
    max_steps = 150,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate = 1e-5,
    logging_steps = 10,
    save_steps = 150,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'cosine',
    report_to = 'none',
    num_generations = 6,
    max_completion_length = 256,
    temperature = 0.7,
    gradient_checkpointing = True,
    fp16 = torch.cuda.is_available(),
    bf16 = False,
    seed = 3407,
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    reward_funcs = compute_reward,
    processing_class = tokenizer,
)

print('Starting GRPO Training...')
print('Expected: 150 steps x 6 gens on Kaggle T4 x2 (~20-25 min)')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 8 — Save Trained Adapters (Fast Kaggle Output)
print('Saving LoRA adapters...')
model.save_pretrained('./math_grpo_adapters')
tokenizer.save_pretrained('./math_grpo_adapters')
print('Adapters saved at ./math_grpo_adapters')

In [ ]:
# Cell 9 — Post-Training Evaluation and Accuracy Improvement
FastLanguageModel.for_inference(model)

post_correct, post_score = evaluate_model('Post-Training Evaluation After 150 GRPO Steps')
print('\nAccuracy Summary')
print('=' * 60)
print(f'Before training: {baseline_correct}/{len(EVAL_CASES)} ({baseline_score:.1f}%)')
print(f'After training:  {post_correct}/{len(EVAL_CASES)} ({post_score:.1f}%)')
print(f'Improvement:     {post_score - baseline_score:+.1f} percentage points')